<img src="images/Logo_UCLL_ENG_RGB.png" style="background-color:white;" />

AI Applications

Lecturer: Jan D'Espallier

Academic year 2026-2027

Inspired by: lecture from Chidi Nweke 2022-2023 & Keras tutorials

# Neural Networks in action.

<center> Artificial Neural networks (ANN) </center>
<br>
<center><img src="images/neural network.webp" style="background-color:white; max-width:35%"></center>

The purpose of this notebook is that you set some first steps in training a Neural net. Training neural networks requires knowledge about data preparation, over-fitting, under-fitting, basic notions of stochastic gradient descent, validation, ... Hopefully you will get a better feeling about what that actually all means.

Note : These days, using existing/pre-designed/pre-trained neural networks is more easy and often more performing than training a model yourself using ```sci-kit learn```.    However some things can feel like magic if you are unclear about some of the fundamentals. That is okay, just like we covered in stochastic gradient descent, learning is an iterative process. We will attempt to give you the basics in the following sessions and from then it's up to you to try things out, make mistakes, find the answers by asking us/reading the documentation, correcting them and trying again.

caveat : It may very well be that not all of you will reach the end of this notebook by this workshop. No worries. You will get the fully completed notebook afterwards..such that can can play around with it @ home.

## How this notebook works..
In this notebook you'll find cells that start with a ❓. These cells contain a question or a task.
The text in the cell contains at the end an indication of the type of content that is expected in the next cell.
- 'txt' : the next cell is a 'mark down' cell, in which you should write in text , your answer to the question.
- 'code': the next cell is a 'code' cell, in which you should write your code an execute it.
- 'exec' : there is no next cell, just execute the task.
Do not remove the answer-cells behind the task-cells. The cells are needed as they exist to evaluate your submission.

Lets start with an easy one:

##### ❓ Write in the cell below your ucll email adress. (txt)

REPLACE_WITH_YOUR_UCLL_EMAIL

## Getting your environment ready
This notebook has been test in an anaconda environment on windows with python 3.11.9 installed

It should run as well on Linux and or with other versions of python as long as the correct packages have been installed.
pip install -r requirements should do the job

##### ❓ Install all packages in your environment. (exec)

Installing the packages manually is also an option. 

- pip install jupyter
- pip install tensorflow
- pip install matplotlib
- pip install scikit-learn
- pip install seaborn

For the last part on hyperprarmeter tuning  
- pip install optuna
- pip install plotly

In [ ]:
import keras
from keras.utils import plot_model
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix # https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split
import seaborn as sns

## Introduction to Tensorflow

Tensorflow gets its name from  something from Mathematics. A tensor is simply a N-dimensional matrix. Color images are an example of a tensor, each pixel has R, G and B values. Alternatives to Tensorflow are Pytorch and MXnet.

<center><img src= "images/tensor.png" style="background-color:white; max-width:50%"></center>

```Keras``` is the high-level API of Tensorflow and what we recommend you use in this course. ```Keras``` contains two API's, the ```Sequential``` API and the ```Functional``` API. You're free to use whichever you prefer the most. The ```Sequential``` API tends to be a bit easier, but with slightly less flexibility. Finally there is also model subclassing.

## The data
We will start this on a very simple dataset called the MNIST handwritten digit classification dataset.
Fortunately, keras exposes some functions to easily load this dataset for experiments.
The function splits the data immediately into a training and a test set. ( it always uses a 60/10 split )

In [ ]:
mnist = keras.datasets.mnist # We load our data
(x_train, y_train),(x_test, y_test) = mnist.load_data() 

Lets look at the format of the images.

In [ ]:
x_train[0].shape

The images only have 1 channel, so they are in gray-scale.

Let's look at some examples to get a feeling about our data.

In [ ]:
# Create a 5x5 grid of subplots
fig, axes = plt.subplots(nrows=5, ncols=5, figsize=(10, 10))

# Adjust the spacing
fig.subplots_adjust(hspace=0.5, wspace=0.5)

# Plot the first 25 images
for i, ax in enumerate(axes.flat):
    # Display an image
    ax.imshow(x_train[i])
    #ax.imshow(x_train[i], cmap='gray')
    
    # Title with the corresponding digit
    ax.set_title(f"Digit: {y_train[i]}")
    
    # Remove x and y axis ticks
    ax.set_xticks([])
    ax.set_yticks([])

# Show the plot
plt.show()

In a real life scenario, it is advised to carefully look at **all** the data presented. In most case, you will be supplied with a lot of rubbish.
The first step in improving the performance of you r network is to clean everything carefully. Here...we are lucky to have a clean db already. :-)

Let's split the training set again in a training set and a validation set ( although you could have the keras fit function do that for you as well)

In [ ]:
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.2, random_state=42)

##### ❓ How many samples dow we have now for each set? (code)

In [ ]:
nb_train_samples = len(x_train)
nb_val_samples = len(x_val)
nb_test_samples = len(x_test)

In [ ]:
# do not change this cell
print({
    "nb_train_samples": nb_train_samples,
    "nb_val_samples": nb_val_samples,
    "nb_test_samples": nb_test_samples
})

In [ ]:
# Check the range of values in x_train
print ('max value in training set:'+str(np.amax(x_train)))

# Check the range of values in y_train
print ('max value in training set:'+str(np.amax(y_train)))

In [ ]:
# neural nets perform best when input is normalised
x_train, x_test, x_val = x_train / 255.0, x_test / 255.0 ,x_val / 255.0 # Scale the data to be between 0 and 1

##### ❓ Use plt.imshow to plot image. (code)

In [ ]:
plt.imshow(x_train[0])

 ##### ❓ Does it still look the same? (txt)

Yes. It's the same. Note : the coloring scheme contains yelow becuase the plt.imshow does this be default for gray scale images.

When working with neural nets, lots of things happen at random. (e.g weight initialization). In order to make your results reproducible during development...you can/have to 'seed' the random generators. 

In [ ]:
# Check the range of values in x_val
print ('max value in validation training set:'+str(np.amax(x_val)))

In [ ]:
np.random.seed(42)
keras.utils.set_random_seed(42)

## Sequential API

### Try 1: Sequential API first example

The sequential API is the most straightforward way to define models in Tensorflow Keras. The first class of importance is the ```Sequential``` class. This takes a list of ```Layers```. 

Considering our image comes in a grid of 28x28 we need to flatten it into a 1D vector. We can use ```Flatten``` for that.

The most important layer to understand is the ```Dense``` layer. A ```Dense``` layer is a layer where each neuron is connected to each input. Within a dense layer you need to specify how many neurons (each neuron is one linear regression) are and what activation function (see: advanced modeling part 1) you want to use. For classification we recommend that you have hidden layers with ```activation='relu```.

When designing a classifier, the final layer often has 'softmax' as activation.  This activation ensures that **all ouputs sum up to 1**. In this way will the outputs represent the 'probability' that an image belongs to a certain class.

This link contains a full list of possible layers Keras offers. https://keras.io/api/layers/ 

In [ ]:
firstModel = keras.models.Sequential([
  keras.layers.Input(shape=(28, 28)),  # Define the input shape explicitly here
  keras.layers.Flatten(),
  keras.layers.Dense(units=128, activation='relu'),
  keras.layers.Dense(units=128, activation='relu'),
  keras.layers.Dense(units=10, activation='softmax')
])

In [ ]:
#An alternative way to program this would be
#firstModel = keras.models.Sequential()
#firstModel.add(keras.layers.Flatten(input_shape=[28, 28]))
#firstModel.add(keras.layers.Dense(128, activation="relu"))
#firstModel.add(keras.layers.Dense(128, activation="relu"))
#firstModel.add(keras.layers.Dense(10, activation="softmax"))

##### ❓ Describe how this neural network looks like. (code)
Use model.summary() to visualise the model architecture in a textual way..

In [ ]:
firstModel.summary()

If you would install graphviz and pydot you could generate an image from your model architecture using plotmodel(model,show_shapes=True, show_layer_names=True)  (optional)

In [ ]:
plot_model(firstModel, show_shapes=True, show_layer_names=True)

##### ❓  What is the network doing? What do the numbers mean?. (txt)

The network takes a 28×28 grayscale image, flattens it into 784 input values, passes it through two hidden Dense layers of 128 ReLU neurons, and ends with 10 softmax outputs representing the probabilities of digits 0–9. The parameter counts are the trainable weights and biases.

Before we can train our model we need to add an optimizer (remember: stochastic gradient descent), a loss function (the error the model uses to evaluate itself) and a metric (a human understandable metric).

We can add all of these by calling the compile method on the model.

💡 We recommend that you use Adam (variant of SGD) as an optimizer with a learning rate of specifically 3e-4, this is a convention / good starting point that is frequently used. For classification [categorical cross entropy](https://youtu.be/rkULCW_h09k) is used as a loss function.  You can tune this later if you want to. We use the 'sparse'variant as we present integers as labels...and not 'one hot encodings].

Full reference of losses: https://keras.io/api/losses/

Full reference of optimizers: https://keras.io/api/optimizers/



In [ ]:
firstModel.compile(optimizer=keras.optimizers.Adam(learning_rate=3e-4),
                   loss='sparse_categorical_crossentropy',
                   metrics=['accuracy'])

To train a model we can use .fit(), as we know from scikit-learn.

There are a number of parameters we definitely need to supply to the .fit() method:


*batch_size*: The amount of training examples that are used to calculate the gradient.

High(er) = more memory needed + more potential to over-fit, but faster training.

Low(er) = less memory needed + less potential to over-fit, but slower training.

1 = SGD (stochastic gradient descent): after every sample, the gradient is calculated and the weights are adapted.


*epochs*: The amount of iterations of gradient descent.


Keras also provides a handy feature to make a validation set called validation_split. If you set this parameter to 0.2, 20% of the dataset is split off. After each epoch, the performance is tested on this set. This can be used to check over-fitting during training. If the loss on validation is a lot higher than on training, you are over-fitting.

💡 Tip: You can always interrupt a notebook cell that is training a network and use the model in subsequent cells. You will have the model with the weights from the latest completed training step.

Full reference for compile() / fit():
https://keras.io/api/models/model_training_apis/

Let's now train (aka fit) the model.

In [ ]:
history = firstModel.fit(x_train[:1000], y_train[:1000], epochs=25, batch_size=10, validation_data=(x_val[:200], y_val[:200]))

```Keras``` also provides handy features to make a validation set called ```validation_split```. If you set this parameter to 0.2, 20 % of the dataset is split off.

In [ ]:
# alternative is to let keras split the dataset into a training and a validation set
# history = firstModel.fit(x_train[:1000], y_train[:1000], epochs=25, batch_size = 10, validation_split=0.2)
# we used the method above to have better control on which data is used for training and validation...

It is very important to look at what happened during the training. 
As we will have to do this often in this notebook, let's create a function to visualise the training history.

In [ ]:
def plotTrainingHistory(history):
    # Create a figure and a set of subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))  # Adjusted figsize to better fit two subplots
    
    # Plot training and validation loss on the first subplot
    ax1.plot(history.history['loss'], label='train')
    ax1.plot(history.history['val_loss'], label='val')
    ax1.set_title('Model Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    
    # Plot training and validation accuracy on the second subplot
    ax2.plot(history.history['accuracy'], label='train')
    ax2.plot(history.history['val_accuracy'], label='val')
    ax2.set_title('Model Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    
    # Improve layout to prevent overlap
    plt.tight_layout()
    
    # Display the plot
    plt.show()

##### ❓ Use this function to plot the history. (code)

In [ ]:
plotTrainingHistory(history)

##### ❓ How does the trend look like for validation and training? (text)

Training performance improves continuously, while validation performance improves at first and then tends to level off or worsen. The growing gap is evidence of overfitting.

First we want to make some prediction with the model. For once (and then never again (why not?)) we use the training samples to evaluate our performance.

In [ ]:
y_train_pred=np.argmax(firstModel.predict(x_train[:1000]),axis=1)

As we will want to calculate model performance (accuracy and f1) multiple times in this notebook, we also make a function to do this for us.

In [ ]:
def displayPerformanceFigures(real,pred,title,includeCF=True):
   if includeCF:
       # Compute confusion matrix
       cm = confusion_matrix(real, pred)
       sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
       plt.xlabel('Predicted labels')
       plt.ylabel('True labels')
       plt.title('Confusion Matrix for '+title)
       plt.show() 

   acc= accuracy_score(real, pred)
   f1 = f1_score(real, pred, average='macro')

   performance = {
        "acc": round(acc, 3),
        "f1": round(f1, 3),
   }
   print(performance)

##### ❓ Run the function we just made.(code)
Plot also the confusion matrix. 
For once you can use (the first 1000 samples of the training data...but never use the training samples again afterwards, to evaluate performance! 

In [ ]:
displayPerformanceFigures(y_train[:1000], y_train_pred, 'training samples', includeCF=True)

##### ❓ How well is the model performing? (txt)

It performs very well on the training samples, but that is not a reliable measure of generalisation because those samples were used for training. Validation and test performance are more meaningful.

##### ❓ Do the same evaluation as above but now on the validation data. (includeCF=True) (code)

In [ ]:
y_val_pred = np.argmax(firstModel.predict(x_val), axis=1)
displayPerformanceFigures(y_val, y_val_pred, 'validation samples', includeCF=True)

##### ❓ Do the same evaluation as above but now on the test data. (includeCF=True) (code)
After all, only the test data is the only data that has never been used during the model training.

In [ ]:
y_test_pred = np.argmax(firstModel.predict(x_test), axis=1)
displayPerformanceFigures(y_test, y_test_pred, 'test samples', includeCF=True)

Option: You could also use use model.evaluate() to check the performance on the test set. 
This method will make a prediction and an accuracy evaluation in 1 shot. An example below on the test set.

In [ ]:
firstModel.metrics_names
firstModel.evaluate(x_test, y_test)

##### How can we improve the performance and reduce the overfitting?

### Try 2 : More data

Same model architecture, but use more data to train :-). Redefine (and recompile) the model..to ensure weight re-initialisation.(otherwise we would start with a trained model..which would give an unfair comparison.

In [ ]:
firstModel = keras.models.Sequential([
  keras.layers.Input(shape=(28, 28)),  # Define the input shape explicitly here
  keras.layers.Flatten(),
  keras.layers.Dense(units=128, activation='relu'),
  keras.layers.Dense(units=128, activation='relu'),
  keras.layers.Dense(units=10, activation='softmax')
])

firstModel.compile(optimizer=keras.optimizers.Adam(learning_rate=3e-4),
                   loss='sparse_categorical_crossentropy',
                   metrics=['accuracy'])

In [ ]:
history = firstModel.fit(x_train, y_train, epochs=15, batch_size=10, validation_data=(x_val, y_val))

In [ ]:
plotTrainingHistory(history)

##### ❓ Does the training history look better when we use more data?

Yes. More training data generally improves validation/generalisation performance and reduces overfitting because the model sees more examples of the underlying patterns.

Lets look at the prediction performance on the test data.

In [ ]:
y_test_pred=np.argmax(firstModel.predict(x_test),axis=1)
displayPerformanceFigures(y_test,y_test_pred,'test samples',includeCF=True)

##### What if we do not have 'more data'? 

There are many tools available to apply **'data augmentation'**
Data augmentation can be done in different forms :
- slight rotation of the data
- horizontal image flipping (would that work here?)

The same label (y) can be used although the x value changes (rotated). So we end up having more data !

This augmentation can be applied in keras by inserting specific layers for this purpose.
  keras.layers.RandomFlip("horizontal"),  # Randomly flips each image horizontally during training
  keras.layers.RandomRotation(0.05),      # Randomly rotates each image during training

##### ❓ Will that work in this case ?  

Horizontal flipping is generally not appropriate for MNIST digits because it can change the structure or meaning of a digit. Small rotations are much more suitable augmentation.

Let's try it out with the code below

In [ ]:
firstModelwithDA = keras.models.Sequential([
  keras.layers.Input(shape=(28, 28)),  # Define the input shape explicitly here
  keras.layers.RandomFlip("horizontal"),  # Randomly flips each image horizontally during training
  keras.layers.RandomRotation(0.05),  # Randomly rotates each image during training
  keras.layers.Flatten(),
  keras.layers.Dense(units=128, activation='relu'),
  keras.layers.Dense(units=128, activation='relu'),
  keras.layers.Dense(units=10, activation='softmax')
])
firstModelwithDA.compile(optimizer=keras.optimizers.Adam(learning_rate=3e-4),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
history = firstModelwithDA.fit(x_train[:1000], y_train[:1000], epochs=25, batch_size=10, validation_data=(x_val[:200], y_val[:200]))

In [ ]:
plotTrainingHistory(history)  

In [ ]:
y_val_pred=np.argmax(firstModelwithDA.predict(x_val),axis=1)

In [ ]:
displayPerformanceFigures(y_val,y_val_pred,'validation samples on model trained with DA',includeCF=True)

##### ❓What do you see in the performance, now that we use data augmentation ? (txt)

...

There is another technique that you know, that can help you to assess potential performance of your model in case you have little amount of data.

Applying the K-Crossfold technique would give you K iterations to evaluate model performance potential. Voting over all models..will improve the performance. We will not elaborate on that technique here.
<center><img src= "images/crossfold.jpg" style="background-color:white; max-width:50%"></center>

##### How can we further reduce overfitting?

### Try 3 : reduce number of trainable parameters

We could make the model smaller by reducing the amount of neurons in the first layer.

💡 A common convention is to have the neurons be powers of 2 so 2, 4, 8, 16, 32, ...



In [ ]:
np.random.seed(42)
keras.utils.set_random_seed(42)

##### ❓ Make a model with 64 in the first and 32 neurons in the second Dense layer. For the rest the same as firstModel.(code)

In [ ]:
secondModel = keras.models.Sequential([
  keras.layers.Input(shape=(28, 28)),
  keras.layers.Flatten(),
  keras.layers.Dense(64, activation='relu'),
  keras.layers.Dense(32, activation='relu'),
  keras.layers.Dense(10, activation='softmax')
])

##### ❓ Compile , fit, plot history and evaluate performance of this smaller model. (code)
As we did above. Make sure you use the testset! And use the same displayPerformanceFigures() function

In [ ]:
secondModel.compile(optimizer=keras.optimizers.Adam(learning_rate=3e-4), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history = secondModel.fit(x_train, y_train, epochs=15, batch_size=10, validation_data=(x_val, y_val))
plotTrainingHistory(history)
y_test_pred = np.argmax(secondModel.predict(x_test), axis=1)
displayPerformanceFigures(y_test, y_test_pred, 'test samples on smaller model', includeCF=True)

##### ❓ How well is the model performing? How does the trend look like for validation and training? Do we need more epochs? (code)

The smaller model still gives strong performance with fewer parameters. Some overfitting can remain. More epochs are useful only while validation loss is still improving; otherwise they mainly increase overfitting.

It is very possible that your accuracy metric and your loss aren't completely aligned, you can improve in one while getting worse in the other. 
You can see that we spend a lot of training time (epochs) without improvement on the validation loss. This extra effort is a waste of time in fact, and the performance kept on dropping. How can we make a better model more efficiently?

### Try 4 : Early stopping

We can do **early stopping**, we train a model and as soon as the validation loss rises for a set amount of iterations we stop the training and go back to the epoch that had the best loss.

<center><img src= "images/early stopping.png" style="background-color:white; max-width:50%"></center>

In [ ]:
np.random.seed(42)
keras.utils.set_random_seed(42)

In [ ]:
# to ensure weigth re-initialisation , we first redefine the model ( note: just recompiling it does not reset the weights!)
secondModel = keras.models.Sequential([
  keras.layers.Input(shape=(28, 28)),  
  keras.layers.Flatten(),
  keras.layers.Dense(64, activation='relu'),
  keras.layers.Dense(32, activation='relu'),
  keras.layers.Dense(10, activation='softmax')
])

Let's again compile and train this model ..then evaluate it's performance on the test data.
As we are lazy...(and good programmers are!) we define a function which we will use more often later on. This avoids some type work..and errors.

Notice the callback for early stopping!

In [ ]:
def compile_and_train_model(mod,nbepochs=25,pat=3):
    mod.compile(optimizer=keras.optimizers.Adam(learning_rate=3e-4),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
    callback = keras.callbacks.EarlyStopping(monitor='val_loss', patience=pat,restore_best_weights=True) # If the validation loss rises for 3 epochs training is stopped..and the weigths of the model with the lowest validation loss are restored.
    history = mod.fit(x_train, y_train, epochs=nbepochs, batch_size=10, validation_data=(x_val, y_val), callbacks=callback)
    return history

##### ❓ Call this function on the second model (code)

In [ ]:
history = compile_and_train_model(secondModel)

❓ Plot the training history (code)

In [ ]:
plotTrainingHistory(history)

##### ❓  Did the training complete faster, with early stopping? (text)

Yes. Early stopping stops once validation loss stops improving for the chosen patience, so it normally uses fewer epochs and saves training time while keeping the best validation weights.

##### ❓ Evaluate as we did before (code)
Use test samples and the same functions.

In [ ]:
y_test_pred = np.argmax(secondModel.predict(x_test), axis=1)
displayPerformanceFigures(y_test, y_test_pred, 'test samples after early stopping', includeCF=True)

##### ❓  What is the danger of early stopping? (text)

If the patience is too small, training can stop too early and the model may underfit. A poor validation set can also cause an inappropriate stopping decision.

### Try 5: Include regularisation 
Higher up, we tried to use more data and we tried to reduce the number of trainable parameters to reduce overfitting.
There is another way (which you may have tried before) to tackle overfitting : regularisation.

Neural networks can also use regularization (confer the (C) parameter in support-vector machines.)

One way to regularise neural nets is ```Dropout``` layers. When training a neural network dropout randomly switches off a percentage of neurons in a layer during each training step. This forces the network not to rely too much on a few specific neurons.
As a result, it learns more robust patterns that still work when some information is missing.
**It has been proven that this is a form of regularization.** It is the easiest one you can implement. You simply put a dropout layer between each dense layer.

<center><img src= "images/dropout.png" style="background-color:white; max-width:50%"></center>


In [ ]:
keras.layers.Dropout(0.2)

Drop out tends to reduce over-fitting, the training loss become significantly higher. By switching off neurons you're forced to learn a more general pattern. Dropout allows you to train larger networks while suffering less from over-fitting.

##### ❓ Create a bigger model..but with drop out regularisation (code)
Increase the size of the first two layers of fifthModel to 128 and 128. (like in our first model)

Insert the drop out layer from above after the 1st and second dense layer.

Store in a variable called sixthModel.

Train the model.

In [ ]:
sixthModel = keras.models.Sequential([
  keras.layers.Input(shape=(28, 28)),
  keras.layers.Flatten(),
  keras.layers.Dense(128, activation='relu'),
  keras.layers.Dropout(0.2),
  keras.layers.Dense(128, activation='relu'),
  keras.layers.Dropout(0.2),
  keras.layers.Dense(10, activation='softmax')
])
history = compile_and_train_model(sixthModel)

##### ❓ Plot the training history and the performance metrics. (code) 

In [ ]:
plotTrainingHistory(history)
y_test_pred=np.argmax(sixthModel.predict(x_test),axis=1)
displayPerformanceFigures(y_test,y_test_pred,'test samples on early stopped model',includeCF=True)

##### ❓ Comment on the model's performance and overfitting, now that we introduced drop out. (text)

We have the best performance so far, but we can still see we are overfitting. We can improve by increasing the dropout rate, this causes the model to overfit less. 

##### ❓ Increase the drop out to 0.5, recompile, set patience to 6 and train for 40 epochs. (code)

In [ ]:
seventhModel = keras.models.Sequential([
  keras.layers.Input(shape=(28, 28)),
  keras.layers.Flatten(),
  keras.layers.Dense(128, activation='relu'),
  keras.layers.Dropout(0.5),
  keras.layers.Dense(128, activation='relu'),
  keras.layers.Dropout(0.5),
  keras.layers.Dense(10, activation='softmax')
])
history = compile_and_train_model(seventhModel, nbepochs=40, pat=6)

❓ Plot the training history and the performance metrics.(no confusion matrix is needed anymore) (code)

In [ ]:
plotTrainingHistory(history)
y_test_pred = np.argmax(seventhModel.predict(x_test), axis=1)
displayPerformanceFigures(y_test, y_test_pred, 'test samples on strong dropout model', includeCF=False)

There is a second popular way to regularise : L2 regularisation.
If you want to see how its done in keras see [here](https://keras.io/api/layers/regularizers/)

### Try 6 : Embedding preprocessing in the model

In [ ]:
mnist = keras.datasets.mnist # We load our data
(x_train, y_train),(x_test, y_test) = mnist.load_data() # We split it into test and train

In [ ]:
eightModel = keras.models.Sequential([
  keras.layers.Input(shape=(28, 28)),  
  keras.layers.Flatten(),
  keras.layers.Rescaling(scale=1./127.5, offset=-1), # Ensures the pixels are between -1 and 1
  keras.layers.Dense(128, activation='relu'),
  keras.layers.Dropout(0.2),
  keras.layers.Dense(128, activation='relu'),
  keras.layers.Dropout(0.2),
  keras.layers.Dense(10, activation='softmax')
])

##### ❓ Compile and train it. (code)

In [ ]:
eightModel.compile(optimizer=keras.optimizers.Adam(learning_rate=3e-4), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history = eightModel.fit(x_train, y_train, epochs=15, batch_size=10, validation_split=0.2)

##### ❓ Plot the history. (code)

In [ ]:
plotTrainingHistory(history)

## Functional API 

With the sequential API we can only define a fairly simple network architecture. 
The layers follow each other 'sequentially'

In case we would like to create more fancy architectures e.g 
- with multiple inputs or
- with bypasses etc..or..
then we need a different API : the functional one.
Lets create our second model but this time with the fuctional API...

In [ ]:
inputs = keras.Input(shape=(28,28))
x = keras.layers.Flatten()(inputs)
x = keras.layers.Dense(64, activation=keras.activations.relu)(x) # you can pass in a function or a string
x = keras.layers.Dense(32, activation='relu')(x) # you can pass in an object or a string
outputs = keras.layers.Dense(10, activation="softmax")(x)

In [ ]:
functionalModel = keras.Model(inputs=inputs, outputs=outputs, name="mnist_model")

##### ❓ Visualise the architecture textually to investigate it. (code)

In [ ]:
functionalModel.summary()

##### ❓ How does the functional API model differ from one with the sequential API? (text)

The Sequential API is a simple linear stack of layers. The Functional API explicitly defines connections between inputs, layers and outputs, allowing multiple inputs/outputs, shared layers and bypass or skip connections.

Let's design a network with a 'bypass' to demonstrate the extra functionality of this interface.

In [ ]:
input_  = keras.layers.Input(shape=[28, 28])
flatten = keras.layers.Flatten()(input_)
hidden1 = keras.layers.Dense(2**14, activation="relu")(flatten)
hidden2 = keras.layers.Dense(512, activation='relu')(hidden1)
hidden3 = keras.layers.Dense(28*28, activation='relu')(hidden2)
reshap  = keras.layers.Reshape((28, 28))(hidden3)
concat_ = keras.layers.Concatenate()([input_, reshap])
flatten2= keras.layers.Flatten()(concat_)
output  = keras.layers.Dense(10, activation='softmax')(flatten2)
functionalModelwithBypass = keras.Model(inputs=[input_], outputs=[output] )

In [ ]:
functionalModelwithBypass.summary()

plot_model(functionalModelwithBypass, show_shapes=True, show_layer_names=True) would give use
<center><img src= "images/functionalModelwithBypass.webp" style="background-color:white; max-width:50%"></center>

## Model subclassing
Just for completeness we mention that if you even want to do more complex stuff like e.g. creating youw own layer types (researchers,advanced ai)
you may want to use yet another keras API-> subclassing. A simple example here below.

In [ ]:
class MNISTmodel(keras.Model):
    def __init__(self):
        super().__init__()
        # The order does not matter, you can place them however you want here
        self.flatten = keras.layers.Flatten()
        self.d3 = keras.layers.Dense(10, activation="softmax")
        # here
        self.d1  = keras.layers.Dense(64, activation='relu')
        # here
        self.d2  = keras.layers.Dense(32, activation='relu')

    def call(self, x): # You must implement this method. This is the order in which the neural network does its predictions
        x = self.flatten(x) # It first flattens the input
        x = self.d1(x) # Afterwards it sends it to the first 64 regressions
        x = self.d2(x) # The second 32 regressions
        return self.d3(x) # returns the predictions
subclassModel = MNISTmodel()

Compile and train it. Plot the history.

In [ ]:
history=compile_and_train_model(subclassModel,nbepochs=15)
plotTrainingHistory(history)

## Hyper parameter tuning

There is not a single way to designing and training neural nets. Try and error...is often the way to find a good nb of layers, with a number of neurons that works well for your problem.
Other 'hyper prarameters' (that is what these are called) are
* Activation function
* The amount of drop out layers
* For each drop out layer, the drop out rate
* The optimizer you will use
* The learning rate for the optimizer
* Loss function
* Batch size

The number of potential hyper-parameter combinations to try out is **HUUUUGE**. You could manually change some combinations...and evaluate the performance of your resulting model. 

### Some tips

💡 **Frequently neural networks are too big to train using cross validation and/or grid search.** Even on GPU it would take too long to train large neural networks sequentially and training them in parallel requires expensive GPU's. 

💡 **We do recommend you to change parameters one by one.** Never change two parameters at the same time if you're manually tuning. For example, adding drop out AND increasing the size. Split it up into 2 separate "experiments" to see the effect of each separately.

💡 **Favour smaller models over larger ones**. Think about the final assignment, a large model makes slower predictions and takes longer to train. If two models have near equal performance you should go for the smallest version. TIP: when training neural networks you can start with a large network that overfits and gradually make it smaller and/or add drop-out until it no longer overfits and captures the patterns correctly.

Training neural networks is time-consuming, both in terms of CPU time as in terms of engineer time. Use them where necessary but try simple things first. 

### Optuna

If you do this hyper tuning exercise manually, you may soon loose the overview.
[Optuna](https://optuna.org/) is a package that helps you to get this search for the wholy grale a bit organised.  
Optuna is a sophisticated hyperparameter optimization framework that goes beyond simple trial-and-error or random combinations of parameters. It employs more advanced and intelligent strategies to efficiently find high-performing hyperparameter configurations. 

Below you find some code that searches the hyper parameter space for the network we were using above.

In a nutshell : Optuna can **suggest** (see the trial.suggest functions) an hyperparameter from a range or set that you define.
In a loop it will use these suggested hyper parameters to optimise an objective function defined by you.
When done..there are some nice visualisations of the results.

In [ ]:
import optuna
import plotly
from optuna.visualization import plot_optimization_history
from optuna.visualization import plot_parallel_coordinate
from optuna.visualization import plot_param_importances

In [ ]:
verbosity=True
epochs=20            # keep this small (5) if you only want to test the code
nb_trials= 10        # keep  this small(5) if you only want to test the code

In [ ]:
def make_model( layer_sizes):
    inputs = keras.Input(shape=(28, 28))
    x = keras.layers.Flatten()(inputs)
    x=  keras.layers.Rescaling(scale=1./127.5, offset=-1)(x) # Ensures the pixels are between -1 and 1
    for size in layer_sizes: 
        x = keras.layers.Dense(size, activation='relu')(x) 

    outputs = keras.layers.Dense(units=10, activation="softmax")(x)

    return keras.Model(inputs, outputs)

In [ ]:
def calculate_performance(x_test,y_test,mod):
    y_test_pred=np.argmax(mod.predict(x_test),axis=1)
    #acc= accuracy_score(y_test,y_test_pred)
    #print("Accuracy: ", acc)
    f1 = f1_score(y_test, y_test_pred, average='macro')
    #print("F1 Score: ", f1)
    return f1

In [ ]:
def objective(trial): # we use optuna to finetune the hyper parameters
    # the hyper parameters are 
    nb_layers = trial.suggest_categorical("nb_layers", [2,3])
    layer_sizes=[]
    for i in range(nb_layers) :
        layer_size =trial.suggest_categorical("lay"+str(i), [32,64,128])
        layer_sizes.append(layer_size)

    weigth_init_seed=trial.suggest_int("seed",1,46)

    print('trial '+str(trial.number)+' optuna suggested layer sizes:'+str(layer_sizes)+' and weigth init seed:'+str(weigth_init_seed))
    
    model = make_model(layer_sizes=layer_sizes)
    if verbosity:
        print(model.summary())
        #keras.utils.plot_model(model, show_shapes=True)

    # Set random seed for reproducibility
    np.random.seed(weigth_init_seed)
    
    model.compile(
            optimizer=keras.optimizers.Adam(0.0001),
            loss=keras.losses.sparse_categorical_crossentropy,
            metrics=['accuracy'],
            )
    callback = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3) # If the validation loss rises for 3 epochs training is stopped
    history = model.fit(x_train, y_train, epochs=epochs,callbacks=callback, batch_size=10, validation_data=(x_val, y_val))
    plotTrainingHistory(history)        

    return calculate_performance(x_test,y_test,model)

##### ❓ What are the hyper parameters that are tuned in this objective function? (text)

The number of hidden layers (2 or 3), the size of each hidden layer (32, 64 or 128 neurons), and the random weight-initialisation seed (1–46).

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=nb_trials)
study.best_params 

In [ ]:
plotly.io.renderers.default = "browser"

Lets generate the study plot.

In [ ]:
plot_parallel_coordinate(study)

In case there is no time to run it :The line above gave me :
<center><img src= "images/parralel_coordinate_plot.png" , max-width:50%"></center>

Lets generate the importances plot.

In [ ]:
plot_param_importances(study)

In case there is no time to run it :The line above gave me :
<center><img src= "images/parameter_importance.png" ></center>

##### ❓ Which seems to be the least important parameter?(text)

The weight-initialisation seed appears to be the least important parameter because it changes the random starting point rather than the model architecture.

## Final take away
These days there are a lot of large 'pre-trained' models available. 
Instead off of designing your own architecture and train it with a (usually) limited set of data that you have access to, it is often much more efficient to fine tune these pretrained networks using the technique of transfer learning.
This will be explained in the next lesson (computer vision). Stay tuned!